# 대화 메모리 (Chat Memory)

- LLM 호출은 기본적으로 이전 호출을 기억하지 않는 stateless 요청이다.
- 대화 맥락을 유지하려면 매 호출에 이전 메시지를 함께 전달해야 한다.
- 이 노트북에서는 LangChain Core의 메시지와 히스토리 인터페이스를 사용해 직접 관리한다.

In [2]:
from dotenv import load_dotenv
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    trim_messages,
)
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
MODEL_NAME = "gemini-3.6-flash"
# 이 모델은 sampling 기본값이 고정되어 있으므로 temperature를 전달하지 않는다.
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)

---

## 메모리 없는 대화

첫 번째 호출의 메시지를 두 번째 호출에 전달하지 않으면 모델은 이전 내용을 알 수 없다.

In [3]:
response1 = llm.invoke([HumanMessage(content="내 이름은 철수야")])
print("응답1:", response1.content)

response2 = llm.invoke([HumanMessage(content="내 이름이 뭐였지?")])
print("응답2:", response2.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


응답1: [{'type': 'text', 'text': '반갑습니다, 철수님! 😊\n\n저는 철수님을 도와드릴 AI 어시스턴트입니다. 오늘 어떤 도움이 필요하신가요? 궁금한 점이나 나누고 싶은 이야기가 있다면 편하게 말씀해 주세요!', 'extras': {'signature': 'EscNCsQNARFNMg8XYKo+QmBGjtJdhSVcIOdxdISGY+hjEbe3qQlySs1ne9jGE81ac7hhRi0LMUnUH0bVcW9Qt7m5OeHFam44S1H0W31O9l0quvj352a7FW//op+wGZGL2JvTDUpg1dA4IsCB3ZIRr5SY/po030uh83E7tYSs+KBh0NVm4XjoURp4h+FhQmg9/i/zLzPSlZkV3tyrAAMpmwWYmVg2G+8FUrV2mEVaDPyAvK1elmFlCgoXa/kBRplHIkDOY+QL0FTUvyY5dPTtAnUHcLAgCra/T3zTrVc0A6pzXRXnI1I/n/SGbzq6dEYv1EAl3INDQYQ10xq5sROGjzVHVNkmk+U1ZJzqAp/oStKFE3/uFH4pewOFH3kCMrjrUxfi7VnGu+/MJV3Vgm7xr4QfzRUjE0dxuXIjGdoC9a6KIlDov4KgnKuzCfkHVzppY6aR0n81WbSRBZRbf9MepN2ORKQgxx9Jay5l0q7SRkEnLlSIXeDdClfzpy27gp2CMQWpjEEoV6hdUj5tqystzS93/HVBl5//vDrtV1cmUivuKooGXz6MA2It4Nz82c4vTMj4RD80770kobUUC9k+lB1xdwTLSXIWmsEHCp+zzOvK0IovPlw9vrfIydqQ1EajkW23sxWz/5UMRR+wO8Cppdy9kTZkYqrJMPOWimpeQv+ACtEIQQ47oPfZH1j4Nk8kbpm80gZ3KXZyz3oVBRwqvvLIo8v7dGNSwREo3jYlrWxxN9XkskQXki7LRrwpmOmAKu7IY03+CPkS3MXiHHXRz4vKv00RWi/FHgVAUyXgjQWWPK5S0/uis1NIWPXyqnUm5a+yS+aYT9T

## 수동으로 히스토리 전달하기

이전 메시지를 리스트에 누적하여 함께 전달하면 맥락이 유지된다.

In [4]:
messages = [
    SystemMessage(content="너는 친절한 상담사야."),
    HumanMessage(content="내 이름은 철수야"),
]
response1 = llm.invoke(messages)

messages.extend([response1, HumanMessage(content="내 이름이 뭐였지?")])
response2 = llm.invoke(messages)
print(response2.content)

[{'type': 'text', 'text': '제가 당연히 기억하고 있지요! 바로 **철수**님이세요. 😊\n\n혹시 저를 살짝 시험해 보신 건가요, 아니면 요즘 생각할 게 많아서 순간 깜빡하셨던 걸까요? \n\n어떤 것이든 괜찮으니, 편안하게 이야기 나누어요. 오늘 하루는 어떻게 보내고 계신가요?', 'extras': {'signature': 'EroKCrcKARFNMg/VTUeRaLye7ZWGzT+9Ql2lmn9XQVdkO7qFa7cB7F17cfBSCZ3L26nYKzXbjLN6BLgY1HF1WaTJDIzdrsOvlpp5swSDmJGX9OcQ/ozZWe75YQhRgnUguBaF+PpW4zOpAHvMM9tuuv12rnNLkaURa9HX7V3B9f0uwuU6Daaei44XsXewlZdAdIX/ueMLWtpVV7MNu4NjD2xhVxkgi4BfxX881Uf5F9r0lEawD7pQaOaTEev/B23qWaQpqXZJ0Ao7qxcZWzir1PlM8hui+Pe3ZsXdcm/5FyEt//cTa2eu3ByYZWqJ3SqqS8bm9SFfoDY9ARpflfDOWEpU+lEH8vd9PL/916eDX+rYCFVL/QA/IIbE4Ky/S7wViitsG9/EVsBPaog/0v4GJqC6c83VC/ZVwaNyfyD8kjDoPciz/IaOBiS1dYY1xBiktH/lQk1V7KNYn5yNuBYbeutKLP0sjrq+6+GbOD6WhHeGAPkUPzpWAql8tqsxGN8GA2A/uc9DJyc1nD/6gUg4DsLy08kpZuQHOmir7dKJw9QDJoGq+uzYuQeRfArePT3sJrtYbC1aGxYu4NY6f8tKswxGmCr3Xq6vESyZqoQGOq6VUwkLm9vMIRamwDvlYcpluk+g7Nyf+jFsTHZIX04m0CBlGfEobQI/cjO00D3rZdPwJV3nIcEYPc9Mr3cqF+PS8S2Nuh7WdR0CxyNFgrlHY/yfpRf1j1z0zQqXZY5IvRMlzyHBx7RTASGG+6mqlLV0fW/g6ovy1tZQoxDR9TAnHP9EwqkZn2Ey/n8Y

---

## InMemoryChatMessageHistory로 관리하기

`InMemoryChatMessageHistory`는 메시지 추가와 조회를 위한 LangChain Core의 기본 인메모리 구현이다. 세션 ID별로 객체를 나누면 여러 대화를 독립적으로 관리할 수 있다.

In [6]:
history_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

def chat(session_id: str, user_input: str) -> str:
    history = get_session_history(session_id)
    user_message = HumanMessage(content=user_input)
    response = llm.invoke([
        SystemMessage(content="너는 친절한 상담사야."),
        *history.messages,
        user_message,
    ])
    history.add_messages([user_message, response])
    return response.content

In [7]:
print(chat("user-1", "내 이름은 철수야"))
print(chat("user-1", "내 이름이 뭐였지?"))

# 다른 세션에는 user-1의 기록이 없다.
print(chat("user-2", "내 이름이 뭐야?"))

[{'type': 'text', 'text': '안녕하세요, 철수 님! 만나서 정말 반가워요. 😊\n\n오늘 저를 찾아와 주셔서 감사해요. 저는 철수 님의 이야기에 귀 기울이고, 편안한 소통을 도와드릴 상담사입니다. \n\n혹시 오늘 어떤 마음으로 저를 찾아오셨나요? 요즈음 고민이 있거나 마음속에 담아둔 이야기가 있다면 무엇이든 편하게 이야기해 주세요. 제가 따뜻하게 들어드릴게요.', 'extras': {'signature': 'EusNCugNARFNMg/VHQ8+f4/1w16vXm6wU6TTfbvmHaXSE4OEDLgIVmK3XFpDfRZAoLhmi40URF5vSzHfgo06quFD+NZQN4rFHkABnmaJSaGcUkPiPSW06Xi6MakPinn9lZpvbrkYgiSzUZbviduhUWwha1qRjn6x6Jx4I7HEV22h817etwQ1WAY28JJUP5jYgKvLVMyB7ZSSGAz73d8NEgn4HvkiiS0177/X0hzrKt571loHcrNhEIe1mbaIumDLaFyVKlLOJjj9KQEauhQTxCaq4fnVI+u15cqa4hfecin3HAkllOgunoVIehWhjGxwYvYJf4fNU0cAKSm/82e3D4U8pqxPRL6My+LHOyhtE8Ok9pxBiP7O+OFfvBJen/We7TpGIeVcGzgFShTWUQp3FqWQQFoZUKyI6B9uYyHtTp+ayjDW/P4loOlVky9+grmDb4WQqDKLymNuGhDnr2Zma8B1rlB3PbITGNwZigPF2gmYh+y4YFHLrcq2k0d6v0vwkjcxWFAuBowRTOZ26ic6ILFKsA0mLQq5G9xrgSVUZ+KUnJgqWdbXd9Xadi8BW5l4/ntxQNoocYkphCsvhex2S0oLKEpdpD1rYvc34hONpTbVc924RU0TJTuUc5K9cNMqCLsBfk6J6oly76xmSRc6M9YJlGqUiUnxmAkg5G2EWhwMGR8z3KpNXzs6jXS80ZEoz18pA8UOMoHGHu0abYY6n9Uew67GDTZVjzMFVBPQ1bb/a41Eu856m3

In [8]:
for message in get_session_history("user-1").messages:
    print(f"[{message.type}] {message.content}")

[human] 내 이름은 철수야
[ai] [{'type': 'text', 'text': '안녕하세요, 철수 님! 만나서 정말 반가워요. 😊\n\n오늘 저를 찾아와 주셔서 감사해요. 저는 철수 님의 이야기에 귀 기울이고, 편안한 소통을 도와드릴 상담사입니다. \n\n혹시 오늘 어떤 마음으로 저를 찾아오셨나요? 요즈음 고민이 있거나 마음속에 담아둔 이야기가 있다면 무엇이든 편하게 이야기해 주세요. 제가 따뜻하게 들어드릴게요.', 'extras': {'signature': 'EusNCugNARFNMg/VHQ8+f4/1w16vXm6wU6TTfbvmHaXSE4OEDLgIVmK3XFpDfRZAoLhmi40URF5vSzHfgo06quFD+NZQN4rFHkABnmaJSaGcUkPiPSW06Xi6MakPinn9lZpvbrkYgiSzUZbviduhUWwha1qRjn6x6Jx4I7HEV22h817etwQ1WAY28JJUP5jYgKvLVMyB7ZSSGAz73d8NEgn4HvkiiS0177/X0hzrKt571loHcrNhEIe1mbaIumDLaFyVKlLOJjj9KQEauhQTxCaq4fnVI+u15cqa4hfecin3HAkllOgunoVIehWhjGxwYvYJf4fNU0cAKSm/82e3D4U8pqxPRL6My+LHOyhtE8Ok9pxBiP7O+OFfvBJen/We7TpGIeVcGzgFShTWUQp3FqWQQFoZUKyI6B9uYyHtTp+ayjDW/P4loOlVky9+grmDb4WQqDKLymNuGhDnr2Zma8B1rlB3PbITGNwZigPF2gmYh+y4YFHLrcq2k0d6v0vwkjcxWFAuBowRTOZ26ic6ILFKsA0mLQq5G9xrgSVUZ+KUnJgqWdbXd9Xadi8BW5l4/ntxQNoocYkphCsvhex2S0oLKEpdpD1rYvc34hONpTbVc924RU0TJTuUc5K9cNMqCLsBfk6J6oly76xmSRc6M9YJlGqUiUnxmAkg5G2EWhwMGR8z3KpNXzs6jXS80ZEoz18pA8UOMoHGHu0abYY6n9Uew67GDTZ

### 인메모리 히스토리의 한계

`InMemoryChatMessageHistory`는 대화 히스토리의 기본 동작을 학습하기에 적합하지만, 프로세스가 종료되면 기록이 사라진다. 이후 LangGraph에서는 같은 흐름을 state와 checkpointer로 관리하며, 데이터베이스 기반 저장소를 연결해 대화를 영속적으로 유지할 수 있다.

---

### Context window 관리

DB에 원본 메시지를 저장하는 것과 모델에 보낼 메시지를 선택하는 것은 별개의 문제다. 대화가 길어지면 비용과 지연 시간이 증가하고 context window를 넘을 수 있다.

| 방법 | 장점 | 단점 |
|---|---|---|
| 메시지 트리밍 | 단순하고 추가 호출 비용이 없음 | 오래된 맥락 유실 |
| 대화 요약 | 오래된 핵심 맥락 압축 | 추가 호출 비용과 요약 오류 가능성 |
| 검색 기반 선택 | 관련 과거 정보만 선택 가능 | 검색·인덱싱 설계 필요 |

요약과 최근 메시지를 함께 유지하는 방식은 유용한 일반 패턴이지만, 서비스 특성과 평가 결과에 맞춰 선택해야 한다.

`trim_messages()`는 전체 히스토리에서 모델에 전달할 메시지만 선택한다. 원본 리스트나 `InMemoryChatMessageHistory`의 메시지를 삭제하지 않고, 조건에 맞게 선택된 새로운 메시지 목록을 반환한다. 따라서 전체 대화는 저장소에 보존하면서 매 호출에 필요한 최근 대화만 모델에 전달할 수 있다.

`trim_messages()`의 주요 옵션은 다음과 같다.

- `max_tokens`: 트리밍 결과에 허용할 최대 토큰 수를 지정한다. 실제로 남는 메시지 수는 각 메시지의 길이에 따라 달라진다
- `strategy`: 앞쪽 또는 뒤쪽 중 어느 메시지를 우선하여 남길지 결정한다
- `token_counter`: 메시지의 토큰 수를 계산할 모델이나 함수를 지정한다
- `include_system`: 첫 system 메시지를 트리밍 결과에 유지할지 결정한다
- `start_on`: 트리밍된 대화가 어떤 역할의 메시지부터 시작해야 하는지 지정한다
- `end_on`: 트리밍된 대화가 어떤 역할의 메시지에서 끝나야 하는지 지정한다

아래 예제에서는 최대 토큰 수를 80으로 제한하고 최근 대화를 우선하여 남긴다. system 메시지는 유지하며, 그다음 대화가 human 메시지부터 시작하도록 설정한다.

처리 흐름은 다음과 같다.

```text
전체 히스토리 저장 → 토큰 수 계산 → 최근 메시지 선택 → 대화 시작 역할 정리 → 모델에 전달
```

트리밍 결과에 오래된 메시지가 포함되지 않으면 모델은 그 내용을 알 수 없다. 중요한 사용자 정보까지 단순히 제거될 수 있으므로, 실제 서비스에서는 최근 메시지와 대화 요약 또는 검색한 과거 정보를 함께 전달하는 방법을 고려한다.

In [ ]:
long_history = [
    SystemMessage(content="너는 친절한 상담사야."),
    HumanMessage(content="안녕하세요"),
    AIMessage(content="안녕하세요. 무엇을 도와드릴까요?"),
    HumanMessage(content="Python에 대해 알려줘"),
    AIMessage(content="Python은 범용 프로그래밍 언어입니다."),
    HumanMessage(content="FastAPI에 대해 알려줘"),
    AIMessage(content="FastAPI는 Python 웹 프레임워크입니다."),
    HumanMessage(content="내 이름은 철수야"),
    AIMessage(content="반가워요, 철수님."),
    HumanMessage(content="내 이름이 뭐였지?"),
]

trimmer = trim_messages(
    max_tokens=80,
    strategy="last",
    token_counter=llm,
    include_system=True,
    start_on="human",
)
trimmed = trimmer.invoke(long_history)
print("원본 메시지 수:", len(long_history))
print("트리밍 후 메시지 수:", len(trimmed))
for message in trimmed:
    print(f"[{message.type}] {message.content}")

---

## 참고: Short-term memory와 long-term memory

두 메모리는 단순히 저장 기간이 아니라 기억을 사용하는 **범위(scope)** 로 구분한다.

| 구분 | Short-term memory | Long-term memory |
|---|---|---|
| 범위 | 현재 대화 세션 | 여러 대화 세션 |
| 저장 내용 | 현재 대화 메시지와 작업 맥락 | 사용자 선호, 프로필, 기억할 사실이나 경험 |
| 예시 | "앞에서 내 이름을 철수라고 말했어" | "이 사용자는 Python 백엔드 개발자다" |

이 노트북에서는 현재 대화의 히스토리인 short-term memory만 다룬다. 세션을 넘어 정보를 저장하고 필요한 기억을 찾는 long-term memory는 이후 과정에서 별도로 다룬다.

---

### 실습문제

**세션별 대화 메모리 만들기**

`InMemoryChatMessageHistory`를 사용하여 세션별로 대화 내용을 기억하는 함수를 만들어보자. 같은 세션에서는 이전 대화를 기억하고, 서로 다른 세션의 대화는 섞이지 않는지 확인한다.

In [9]:
history_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

def chat(session_id: str, user_input: str) -> str:
    history = get_session_history(session_id)
    user_message = HumanMessage(content=user_input)
    response = llm.invoke([
        SystemMessage(content="너는 군인이야."),
        *history.messages,
        user_message,
    ])
    history.add_messages([user_message, response])
    return response.content

print(chat("user-1", "내 이름은 철수야"))
print(chat("user-1", "내 이름이 뭐였지?"))

# 다른 세션에는 user-1의 기록이 없다.
print(chat("user-2", "내 이름이 뭐야?"))

[{'type': 'text', 'text': '**충성! 철수 님, 성함 확인했습니다!**\n\n저는 현재 국방의 의무를 다하고 있는 군인입니다! \n오늘 어떤 임무나 도움이 필요해서 저를 부르셨습니까? 말씀만 하십시오! 이상!', 'extras': {'signature': 'EtIOCs8OARFNMg/0szwzjutKAP3VcRtTStk4Z5HAFSZg/zbERxWiQGTZCcM7Uu14ySK+EVNuRjVRwj6xOOCNTkr9Ih80/qhnQ3fKzquhqH+hTQFShyCLhT6eyZ8xDDH9uPlAz+Z/6JCcIbfmgduCd7PU2xDba2/hbMhesIT5qBtxLWAhdxCog1wxuShTzVklTvxvnrmWBA3wqwmQpWGC1Wg4u8mvCbOJ8FdNQeseAdbSUcPOdFN3yePxOoz3YjI16ZwGhbZxzBwkwTxFl8PARNgY3vBpKjFz8cVgdQUfHQhcxk7Qk1BwEYdzFYRGqDUE0ZjYz1vjGZLwEjVcJtFvpAlRfLCghQ6zKNuYr/bQ5DUOLKLcodly1k+eODH+xcES1nqOAjVNxgD+HBTwUotNmSt5kZuKZ0kR0Mcv/R3IFGVdUgmmO02H6hDbqCwq424DNfwhMhGgWRHmc1mlG4hLsWyJCVJcFwe6Ghu4xEZmN6og4zwY4IH3Q4k3H13RsvQv7QQUX+2ZoKGkmzk/LOdvoixUIV67knHh903JDdZPz5HxnGESke14idNm7fgtb9bHr2AiNqySMi+gNa3HbSWwDl8Kih8nb21xKGCKVy6AywrJ49mzPs0HHgD6rtIT3+a2TWeLAb2bJETZfCXhjUPe6FWLS87pgnTVrLln/3tOGnrK5nYTCwcnC/ij1POdiWM9W7KCUYsVp4eq/JtYodN+Ogda984KtG2/vzE5Su4KS8DsBRZ6Abo4HS6QYSm/N1HQMWEUpUzEW68Yu8d7XQWGQL3zuCT4YtkjG+SP/H2jDJjgwP9dbq3afn5hDhpfRm3yH3zp2PuZNvxO

In [ ]:
import pprint 
# 저장소.
my_history = InMemoryChatMessageHistory()

for _ in range(10):
    user_input = input()
    user_message = HumanMessage(content=user_input)
    
    response = llm.invoke([
        *my_history.messages,
        user_message
    ])
    my_history.add_messages([user_message, response])
    print("response: ", response.content)
    print()
    pprint(my_history.messages)



**최근 대화만 전달하기**

위에서 만든 세션별 대화 함수에 `trim_messages()`를 적용해보자. 전체 대화는 히스토리에 보존하면서 모델에는 최근 대화만 전달한다. 충분히 긴 대화를 실행한 뒤 전체 메시지 수와 모델에 전달된 메시지 수가 달라지는지 확인한다.

In [ ]:
long_history = [
    SystemMessage(content="너는 친절한 상담사야."),
    HumanMessage(content="안녕하세요"),
    AIMessage(content="안녕하세요. 무엇을 도와드릴까요?"),
    HumanMessage(content="Python에 대해 알려줘"),
    AIMessage(content="Python은 범용 프로그래밍 언어입니다."),
    HumanMessage(content="FastAPI에 대해 알려줘"),
    AIMessage(content="FastAPI는 Python 웹 프레임워크입니다."),
    HumanMessage(content="내 이름은 철수야"),
    AIMessage(content="반가워요, 철수님."),
    HumanMessage(content="내 이름이 뭐였지?"),
]

trimmer = trim_messages(
    max_tokens=80,
    strategy="last",
    token_counter=llm,
    include_system=True,
    start_on="human",
)
trimmed = trimmer.invoke(long_history)
print("원본 메시지 수:", len(long_history))
print("트리밍 후 메시지 수:", len(trimmed))
for message in trimmed:
    print(f"[{message.type}] {message.content}")

원본 메시지 수: 10
트리밍 후 메시지 수: 8
[system] 너는 친절한 상담사야.
[human] 안녕하세요
[ai] 안녕하세요. 무엇을 도와드릴까요?
[human] Python에 대해 알려줘
[ai] Python은 범용 프로그래밍 언어입니다.
[human] FastAPI에 대해 알려줘
[ai] FastAPI는 Python 웹 프레임워크입니다.
[human] 내 이름은 철수야


In [17]:
from pprint import pprint

# 저장소.
my_history = InMemoryChatMessageHistory()

trimmer = trim_messages(
    max_tokens=30,
    strategy="last",
    token_counter=llm,
    include_system=True,
    start_on="human",
)

for _ in range(10):
    user_input = input()
    user_message = HumanMessage(content=user_input)

    # 전체 메시지
    all_messages = [
        *my_history.messages,
        user_message
    ]

    # invoke를 하기 전 trim을 할 것이다.
    # 즉, invoke하는 message는 잘린 메시지.
    trimmed_message = trimmer.invoke(all_messages)
    response = llm.invoke(trimmed_message)

    # 단, history에는 모든 기록들이 다 남기도록 할꺼야.
    my_history.add_messages([user_message, response])
    
    print("response: ", response.content)
    print("all history")
    pprint(my_history.messages)
    print('trim_messages')
    pprint(trimmed_message)
    print()



response:  [{'type': 'text', 'text': '안녕하세요! 반갑습니다. 오늘 어떤 도움이 필요하신가요?', 'extras': {'signature': 'EowMCokMARFNMg9QTdAJFfz6LixQT80/cYbdl0yUULZ8RxzuBI+bvtMe3dpqzHIrR5N240t0b8LXollOe8/nGoL/2NOLKMIfFn4erQ/enKH/e0VCp0pw4NKZFeQoiFNt9rAFFWQGkRA/VnBONwg17DWoOHBebA4SQ7kuFFn88XlP/OLXpOg8AolJc1nIwjdHnLf2yiNqzXzsY9TkHPxD6bHSza21Xf9569UyA50wtIYR2QVixaU7X+DaIqGdXwHNw2wUOq7WrvAKA7Znz+FWD+ieV3UpdiyQAqCC71mrdwAgZ5+3R/Y6ui8DosDTLHsvRpNRmHisrtD97V4lwnxtJBU9TXBAxSBM/QFdEV9zC/4eFfSpNeTRIChACs6ljEWj1W0j429GAAEQ5yK1+Jm/szlVuMOeuy6aKK8LgGbq2F2E2Vn2AiTNeWhe1cgKIZaBpFAqxMYVABXwJppPixHtAduJUQHlMWwz7IZtQdUNRv7sbzl8tAQMcRqt2gPolCiktPRZ7qaoQm4yF9shskua9x+sb6CmnLYuF3oL/hBBvkDFH537nvfhfDOAhVV4PHGUiuCdL8iUbZqFyRU1t0w0sZmOPJELKab9vs4f/UPV4HGXaIo+HhkptmVRdeC+uSMl/sbdSKVm5FuSzyhQc/LGxRTtxqVd+1vo5aLkAPi4VkGZPLqdmi1riBq25ABsGxONC+bkBQgEbAM0L4LFfH1OJ78usP+li5q6WRCkCnwGsbywNc+vzUrc3xi2jUmoJmIgjvFaGGOu5adXgbsKlcRnvSGIQ0VvpP5CDonRCnOlw3XoeRKH9WBgw0EcfXv2qskQWqzvxSYMivt5kecKy2l5x/lJAMNzcrErsDjx+FHLBDnt9+to2RUyhy3dyAj5d7a6LslVcH9

c:\Users\snail\.pyenv\pyenv-win\versions\3.13.13\Lib\site-packages\langchain_google_genai\chat_models.py:3756: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


ValueError: contents are required.